In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
import ana_utils

In [ ]:
import filter_utils

In [ ]:
from copy import copy

In [ ]:
obj = "yasone3"

In [ ]:
cat = ana_utils.read_catalogue(obj, filter_bad=False, catname="allcolours")

In [ ]:
cat_psf = ana_utils.read_catalogue(obj, filter_bad=False, catname="allcolours_psf")

In [ ]:
obs_props = ana_utils.get_obs_props(obj)

## Extinction

In [ ]:
from dustmaps.sfd import SFDQuery
from dustmaps.bayestar import BayestarWebQuery
from dustmaps.planck import PlanckQuery

In [ ]:
coords = ana_utils.to_coords(cat)

In [ ]:
def plot_dustmap(dustmap, distance, xymax=6, N=100, label="E(B-V)"):

    ra = np.linspace(-xymax, xymax, N) / 60 / np.cos(np.deg2rad(obs_props["dec"])) + obs_props["ra"]
    dec = np.linspace(-xymax, xymax, N) / 60 + obs_props["dec"]
    ra, dec = np.meshgrid(ra, dec)
    coords = SkyCoord(ra*u.deg, dec*u.deg, distance=distance*u.kpc)


    Av = dustmap(coords)
    plt.imshow(Av, origin="lower", extent=(np.min(ra), np.max(ra), np.min(dec), np.max(dec))
              )
    plt.colorbar(label=label)
    plt.xlabel("ra / deg")
    plt.ylabel("dec / deg")

In [ ]:
ra0 = obs_props["ra"]
dec0 = obs_props["dec"]
ra0, dec0

In [ ]:
coord0 = SkyCoord(ra0, dec0, unit="deg")

In [ ]:
plot_dustmap(SFDQuery(), 12, xymax=3*60)

xl, xh = ra0 - 3/60 / np.cos(np.deg2rad(dec0)), ra0 + 3/60 / np.cos(np.deg2rad(dec0))
yl, yh = dec0 - 3/60, dec0 + 3/60
# plt.plot([xl, xh, xh, xl, xl], [yl, yl, yh, yh, yl])

xl, xh = np.min(cat["R_ALPHA_J2000"]), np.max(cat["R_ALPHA_J2000"])
yl, yh = np.min(cat["R_DELTA_J2000"]), np.max(cat["R_DELTA_J2000"])

plt.plot([xl, xh, xh, xl, xl], [yl, yl, yh, yh, yl], lw=1)


In [ ]:
ra0, dec0

In [ ]:
Ebv = SFDQuery()(coord0)
Ebv, 3.1*Ebv

In [ ]:
plot_dustmap(SFDQuery(), 12)

xl, xh = np.min(cat["R_ALPHA_J2000"]), np.max(cat["R_ALPHA_J2000"])
yl, yh = np.min(cat["R_DELTA_J2000"]), np.max(cat["R_DELTA_J2000"])

plt.plot([xl, xh, xh, xl, xl], [yl, yl, yh, yh, yl], lw=1)


The above plots show the SFD extinction maps around our region. The rectangle is the bounds of the catalog. Over this region, the extinction only typically varies by 0.01 dex, so extinction correction appears to be a smaller effect here.

In [ ]:
# deredden magnitudes

coords = ana_utils.to_coords(cat)
A_V = 3.1 * SFDQuery()(coords)
A_g, A_r, A_i = ana_utils.get_extinction(A_V)

In [ ]:
A_g_mean = np.median(A_g)
A_r_mean = np.median(A_r)
A_i_mean = np.median(A_i)

In [ ]:
A_g_mean, A_r_mean

In [ ]:
0.48035105 + 15.88

extinction differences
- Yasone 1: +0.34 in R, 0.15 in GR, 0.09 in RI
- Yasone 2: +0.35 in R, 0.16 in GR, 0.09 in RI
- Yasone 3: +0.48 in R, 0.21 in GR, 0.12 in RI

In [ ]:
ana_utils.get_extinction(1.0), ana_utils.get_extinction(np.median(A_V))

In [ ]:
delta_A = ana_utils.get_extinction(obs_props["A_V"]) - np.array(ana_utils.get_extinction(np.median(A_V)))
delta_A, delta_A[0] - delta_A[1], delta_A[1] - delta_A[2]

In [ ]:
cat["G_MAG"] -= A_g
cat["R_MAG"] -= A_r
cat["I_MAG"] -= A_i


In [ ]:
plt.scatter(cat["R_MAG"] - cat["I_MAG"] + (A_r - A_i - A_r_mean + A_i_mean) , cat["R_MAG"] + A_r - A_r_mean, s=1, lw=0, label="mean A(v)")
plt.scatter(cat["R_MAG"] - cat["I_MAG"], cat["R_MAG"], s=1, lw=0, label="per-star A(v)")
plt.xlim(-1, 2)
plt.ylim(27, 17)

arya.Legend(-1)
plt.xlabel("R - I")
plt.ylabel("I")

In [ ]:
plt.scatter(cat["G_MAG"] - cat["R_MAG"] + (A_g - A_r - A_g_mean + A_r_mean) , cat["G_MAG"] + A_g - A_g_mean, s=1, lw=0, label="Mean A(v)")
plt.scatter(cat["G_MAG"] - cat["R_MAG"], cat["G_MAG"], s=1, lw=0, label="per-star A_v")

plt.xlim(-1, 2)
plt.ylim(27, 17)

arya.Legend(-1)
plt.xlabel("G - R")
plt.ylabel("G")

## Mangitude setup

Since only the default magnitude columns (G_MAG, R_MAG, and I_MAG) are calibrated, 
we simply use these columns to calibrate other magnitude systems. The other columns do not
have a zeropoint offset. For simplicity, I use stars of magnitudes 19 to 22 

In [ ]:
def calibrate_mag_col(cat, col):
    for filt in ["G", "R", "I"]:
        filt_good = cat[f"{filt}_MAG"] > 19
        filt_good &= cat[f"{filt}_MAG"] < 22
        filt_good &= ~cat[f"{filt}_MAG"].mask
        filt_good &= cat[f"{filt}_FLAGS"] == 0
        if hasattr(cat[f"{filt}_{col}"], "mask"):
            filt_good &= ~cat[f"{filt}_{col}"].mask
            
        residual = np.median((cat[f"{filt}_{col}"] - cat[f"{filt}_MAG"])[filt_good])

        print(residual)
        cat[f"{filt}_{col}"] -= residual

In [ ]:
mag_choices = ["MAG", "MAG_APER_0", "MAG_APER_1", "MAG_APER_2", "MAG_APER_3", "MAG_APER_4", "MAG_APER_5", "MAG_APER_6", "MAG_AUTO", "MAG_PSF", "MAG_WIN", "MAG_ISO"]
mag_choices = np.array(mag_choices)

mag_choices = mag_choices[np.isin("R_" + mag_choices, cat.colnames)]

In [ ]:
calibrate_mag_col(cat_psf, "MAG_PSF")

In [ ]:
for mag in mag_choices:
    calibrate_mag_col(cat, mag)


In [ ]:
col = "MAG_WIN"
for filt in ["G", "R", "I"]:
    plt.scatter(cat[f"{filt}_MAG"], cat[f"{filt}_{col}"] - cat[f"{filt}_MAG"], s=1, lw=0, color="k", alpha=0.3)
    plt.axhline(0, color="C1")
    plt.ylim(-1, 1)
    plt.xlim(12, 30)
    plt.xlabel(f"default magnitude {filt}")
    plt.ylabel(f"calibrated windowed magnitude - default")
    plt.show()

In [ ]:
col = "MAG_PSF"
for filt in ["G", "R", "I"]:
    plt.scatter(cat_psf[f"{filt}_MAG"], cat_psf[f"{filt}_{col}"] - cat_psf[f"{filt}_MAG"], s=1, lw=0, color="k", alpha=0.3)
    plt.axhline(0, color="C1")
    plt.ylim(-1, 1)
    plt.xlim(12, 30)
    plt.xlabel(f"default magnitude {filt}")
    plt.ylabel(f"calibrated psf magnitude - default")
    plt.show()

The above plots show an example of this calibration for the Source Extractor windowed magnitudes. In general, this naive method works reasonably well

## Star Galaxy Seperation

For star-galaxy seperation, we use the difference in the 3px and 5px aperture magnitudes. 
3px is about the typical FWHM of good point-like sources, so sources with a larger increase in the magnitudes between 3px and 5px
are likely galaxies. However, this method is strongly biased by poor-background behaviour (nearby bright sources).

In [ ]:
aperture_radii = [1, 2, 3, 5, 7, 10, 20]

In [ ]:
aperture_radii[2]

In [ ]:
from astropy.stats import sigma_clipped_stats

In [ ]:
def filter_not_bright_point(cat):
    filt_good = cat["R_FLUX_RADIUS"] < 3
    filt_good &= cat["R_MAG"] < 21
    filt_good &= cat["R_ELLIPTICITY"] < 0.3
    return filt_good

In [ ]:
def derive_threshhold(flux_param, cat, sigma=3):

    filt = filter_not_bright_point(cat)
    
    μ_flux, med_flux, std_flux = sigma_clipped_stats(flux_param[filt], stdfunc="mad_std", sigma=3)

    return med_flux + sigma*std_flux

In [ ]:
mag_diff_thresh = derive_threshhold(cat["MAG_23"], cat)

In [ ]:
cat_good = cat[filter_not_bright_point(cat)]

In [ ]:
def filt_finite(col):
    filt = np.isfinite(col)
    if hasattr(col, "mask"):
        filt &= ~col.mask

    return col[filt]
    

In [ ]:
[np.nanmedian(filt_finite(cat_good[f"{filt}_FWHM_IMAGE"])) for filt in ["G", "R", "I"]]

In [ ]:
[np.nanmedian(filt_finite(cat_good[f"{filt}_FWHM_WORLD"] * 3600)) for filt in ["G", "R", "I"]]

In [ ]:
plt.scatter(cat["R_MAG"], cat["MAG_23"],c=np.log10(cat["R_FWHM_IMAGE"]), vmin=0.3, vmax=1.3, s=1, alpha=1, lw=0)
plt.axhline(mag_diff_thresh, color="k")
plt.ylim(-1, 1)
plt.xlim(18, 27)
plt.xlabel("R mag")
plt.ylabel("R(3pix) - R(5pix) (mag)")
plt.colorbar(label="log fwhm / pix")

In [ ]:
ϵ = 0.003
χ = 3
sg_sep = cat_psf["R_SPREAD_MODEL"] < np.sqrt(ϵ**2 + χ**2 * cat_psf["R_SPREADERR_MODEL"]**2)


In [ ]:
plt.scatter(cat_psf["R_SPREAD_MODEL"], cat_psf["R_MAG"], c=sg_sep, s=1, lw=0)
plt.xlim(-0.05, 0.05)
plt.ylim(27, 16)
plt.xlabel("spread model")
plt.ylabel("r (mag)")


In [ ]:
plt.scatter(cat_psf["MAG_23"], cat_psf["R_SPREAD_MODEL"], s=1, lw=0, alpha=1, c=sg_sep)
plt.ylim(-0.04, 0.04)
plt.xlim(-1, 1.5)
plt.axvline(mag_diff_thresh, color="k")


In [ ]:
plt.scatter(cat_psf["R_MAG"], cat_psf["MAG_23"],c=sg_sep, s=1, alpha=1, lw=0)
plt.axhline(mag_diff_thresh-0.03, color="k")
plt.ylim(-1, 1)
plt.xlim(18, 27)
plt.xlabel("R mag")
plt.ylabel("R(3pix) - R(5pix) (mag)")


The above plot demonstrates that the r23 seperator is nearly as good as SE's spread model

# Bounds

In [ ]:
mask, poly = filter_utils.get_region_mask(obj)

In [ ]:
safe_region = filter_utils.get_sample_region(40/60, mask, poly, coord0)

In [ ]:
points = [filter_utils.sample_from_poly(safe_region) for i in range(10000)]

In [ ]:
plt.scatter(cat["xi"], cat["eta"], s=1, c="k")
plt.scatter([p[0] for p in points], [p[1] for p in points], alpha=0.1)
plt.xlabel("xi")
plt.ylabel("eta")